# Exit-Field Fourier Representations and Checks

This optional companion to [9.1](../9_1_Defining_the_Complex_Exit_Field.ipynb) collects the extended Fourier descriptions, one-cell projection formulas, periodicity comparison and exact three-order verification. Read the single-grating explanation in 9.1 first; [9.2](../9_2_Sampling_Complex_Fields_and_Their_Spectra.ipynb) handles numerical sampling.

These independent checks use $p=1000$ nm, $A_i=1$ and, for the tilted case, $k_{x,i}p=\pi/2$. The binary comparison has opening fraction $D=0.4$. The final check deliberately switches to a smooth mask with exactly three harmonics, so that a finite order sum can be checked exactly. These examples are separate from the live controls in 9.1.

Run the notebook independently from its setup cell. All calculations prescribe a scalar thin-mask field; no propagation or material-scattering solve is performed.

In [ ]:
from pathlib import Path
import sys
import numpy as np
from IPython.display import display, Image

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / 'wave_optics').is_dir()), None)
if project_root is None:
    raise RuntimeError('Run from the project or one of its subdirectories.')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from wave_optics.exit_field_view import (
    exit_field_periodicity_png, exit_field_reconstruction_png)

p_nm = 1000.0
A_i = 1.0
G = 2*np.pi/p_nm
phase_step = np.pi/2
kx_i = phase_step/p_nm

## 1. Continuous representation: describe the same field by its components

A simple component $e^{ik_xx}$ has a phase that changes uniformly across $x$. Larger $|k_x|$ means faster spatial variation; the sign gives the direction of that phase change. Fourier analysis finds how much of each component builds the field:

$$\boxed{\widetilde U_{\mathrm{exit}}(k_x)
=\int_{-\infty}^{\infty}U_{\mathrm{exit}}(x)e^{-ik_xx}\,dx.}$$

Adding those components back gives

$$U_{\mathrm{exit}}(x)=\frac1{2\pi}\int_{-\infty}^{\infty}
\widetilde U_{\mathrm{exit}}(k_x)e^{ik_xx}\,dk_x.$$

Here $k_x$ is in rad/nm. The FFT appendix also uses $f_x=k_x/(2\pi)$ in cycles/nm; the matching inverse integral then has $e^{i2\pi f_xx}$ and no $1/(2\pi)$ factor.

| Expression | Category | What it describes |
| --- | --- | --- |
| $U_{\mathrm{exit}}(x)$ | Position-space field | Amplitude and phase at a position |
| $\widetilde U_{\mathrm{exit}}(k_x)$ | Continuous spectrum | Component weights indexed by transverse wavevector |
| $\{C_m\}$ | Discrete order-space data, when applicable | A list of component amplitudes and phases |

Fourier analysis changes the **representation**. A physical model is still needed to obtain the field in the first place.

## 2. Discrete representation: a periodic mask

Now also require $t(x+p)=t(x)$. The physical period $p$ introduces $G=2\pi/p$. Its Fourier series is

$$t(x)=\sum_m
\underbrace{t_m}_{\text{mask coefficient}}
\underbrace{e^{imGx}}_{\text{mask harmonic}}.$$

$t(x)$ is the transmission at a position. Each complex $t_m$ describes one component of the whole repeating pattern. Periodicity can be entirely in amplitude: even when $\phi_t(x)=0$, a repeating amplitude pattern introduces $G$.

Insert this series into the thin-mask product $U_{\mathrm{exit}}=tU_i$ with $U_i=A_i e^{ik_{x,i}x}$:

$$U_{\mathrm{exit}}(x)
=\underbrace{\left[\sum_m t_m e^{imGx}\right]}_{\text{periodic mask}}
 \underbrace{\left[A_i e^{ik_{x,i}x}\right]}_{\text{incident plane wave}}.$$

Multiply the incident field into every term:

$$\boxed{U_{\mathrm{exit}}(x)=\sum_m
\underbrace{A_i t_m}_{C_m}
e^{i\underbrace{(k_{x,i}+mG)}_{k_{x,m}}x}.}$$

| Expression | Category | Meaning |
| --- | --- | --- |
| $t_m$ | Mask coefficient | Dimensionless complex weight of a transmission harmonic |
| $C_m=A_i t_m$ | Exit-field coefficient | Field amplitude and phase of order $m$ |
| $k_{x,i}$ | Illumination property | Incident transverse phase rate |
| $mG$ | Grating contribution | Integer harmonic shift set by the mask period |
| $k_{x,m}=k_{x,i}+mG$ | Order location | Total transverse wavevector of that component |

Thus $U_{\mathrm{exit}}(x)=\sum_m C_m e^{ik_{x,m}x}$ is a **position-space reconstruction from order-space data**. It still gives the field as a function of $x$. A Fourier integral extracts components; this sum adds known components back together.

The relation $C_m=A_i t_m$ belongs to this thin-mask example. FMM generally supplies different $C_m$ from its scattering solution.

For a **centred binary mask**, the opening fraction is

$$D=\frac{W_{\mathrm{open}}}{W_{\mathrm{open}}+W_{\mathrm{close}}},\qquad
t_m=D\,\operatorname{sinc}(mD),\qquad
\operatorname{sinc}(u)=\frac{\sin(\pi u)}{\pi u},\quad\operatorname{sinc}(0)=1.$$

Therefore the same baseline field can be written as

$$\boxed{U_{\mathrm{exit}}(x)=t(x)e^{ik_{x,i}x}
=\sum_{m\in\mathbb Z}\underbrace{D\,\operatorname{sinc}(mD)}_{C_m\text{ for }A_i=1}
e^{i\underbrace{(k_{x,i}+mG)}_{k_{x,m}}x}.}$$

For this companion's choice $D=0.4$: $C_0=0.4$, $C_{\pm1}\approx0.30273$, $C_{\pm2}\approx0.09355$, and $C_{\pm3}\approx-0.06237$.

- Changing $d$ at fixed $D$ changes $G$ and the order spacing; these thin-mask coefficients stay the same.
- Changing $D$ at fixed $d$ changes the widths and coefficients; the order locations stay the same.
- Changing only the illumination angle shifts the physical order locations through $k_{x,i}$; these coefficients stay the same **in this prescribed thin-mask model**.

The binary mask has infinitely many nonzero harmonics. The series equals the mask product away from its sharp edges; at an ideal edge it converges to the midpoint value. A finite order sum only approximates the sharp-edged field.


## 3. Extended periodicity comparison: the mask, the illumination, or the exit field?

The mask repeats after $p$, but a tilted plane wave has its own transverse phase period $2\pi/|k_{x,i}|$. Across one mask cell it acquires the phase step $\Delta\phi=k_{x,i}p$:

$$t(x+p)=t(x),\qquad U_i(x+p)=e^{i\Delta\phi}U_i(x),$$

$$\boxed{U_{\mathrm{exit}}(x+p)=e^{i\Delta\phi}U_{\mathrm{exit}}(x).}$$

### Repeated cell drawings can represent Bloch-periodic fields

A sketch that copies the same local wave pattern into every cell does not specify the absolute complex phase of each copy. To identify the boundary condition, state the phase factor between corresponding points.

Let $j$ be the cell number and $\xi$ the position measured from that cell's own origin, so $x=\xi+jp$. Bloch repetition is

$$\boxed{U_{\mathrm{exit}}(\xi+jp)
=\underbrace{e^{ij\Delta\phi}}_{\text{phase attached to cell }j}
 \underbrace{U_{\mathrm{exit}}(\xi)}_{\text{same local profile}},
\qquad \Delta\phi=k_{x,i}p.}$$

| Cell | Local profile | Factor multiplying that profile |
| --- | --- | --- |
| $j=0$ | $U_{\mathrm{exit}}(\xi)$ | $1$ |
| $j=1$ | $U_{\mathrm{exit}}(\xi)$ | $e^{i\Delta\phi}$ |
| $j=2$ | $U_{\mathrm{exit}}(\xi)$ | $e^{2i\Delta\phi}$ |

Thus repeated local strokes with these phase factors and a continuous global plane-wave phase can be two drawings of the **same Bloch-periodic field**. A geometric sketch alone cannot distinguish ordinary PBC from Bloch-PBC.

Ordinary PBC is the special case $e^{i\Delta\phi}=1$: every copy then has the same complex values at corresponding positions. General Bloch-PBC allows a constant phase difference between copies. This is a relation across the computational cell, not a physical wall.

For this side-by-side comparison, keep the **opening fraction $D=0.4$ for this independent check**. Use $p=1000$ nm and unit incident amplitude, and prescribe the phase step directly. The comparison uses the setup values of this companion.

- **A: exact repetition**, defined here by normal incidence. Here normal incidence makes $U_i=1$, so $U_{\mathrm{exit}}=t$. The complete complex field repeats after $p$.
- **B: repetition with a phase offset**, defined here by tilted illumination. The example has $\Delta\phi=\pi/2$. The mask and exit-field magnitude repeat, while the complex exit field advances by a quarter cycle from cell to cell.

The middle row shows **unwrapped incident phase**. The bottom row shows both real and imaginary field components; checking intensity alone would hide their difference. The orange lines mark cell boundaries, with $p$ denoting cell width rather than a cell number.

In [ ]:
display(Image(data=exit_field_periodicity_png(p_nm, A_i, phase_step)))

At $x=0,p,2p,3p$, the chosen mask transmission is always 1. For column B, the incident and exit-field values are therefore $1,i,-1,-i$. Their magnitudes all equal 1, but their phases differ. The complex field repeats after four cells in this example.

### A. Direct analysis when the exit field repeats after p

For normal incidence, $k_{x,i}=0$:

$$U_{\mathrm{exit}}(x)=A_i t(x)=\sum_m C_m e^{imGx},$$

$$C_m=\frac1p\int_0^p U_{\mathrm{exit}}(x)e^{-imGx}\,dx.$$

The direct flow has an explicit condition:

$$\underbrace{t(x)U_i(x)}_{\text{repeating product}}
=\underbrace{U_{\mathrm{exit}}(x)}_{U_{\mathrm{exit}}(x+p)=U_{\mathrm{exit}}(x)}
\xrightarrow{\text{Fourier series over }p} C_m.$$

### B. Analyze the periodic factor for general tilted illumination

Multiplying by the inverse incident phase factor gives

$$U_{\mathrm{exit}}(x)e^{-ik_{x,i}x}=A_i t(x).$$

This is a mathematical change of representation. It preserves the known incident tilt, which is included again during reconstruction:

$$\underbrace{U_{\mathrm{exit}}(x)e^{-ik_{x,i}x}}_{\text{repeats every }p}
\xrightarrow{\text{Fourier series over }p}C_m,$$

$$C_m=\frac1p\int_0^p U_{\mathrm{exit}}(x)
e^{-i(k_{x,i}+mG)x}\,dx,$$

$$U_{\mathrm{exit}}(x)=e^{ik_{x,i}x}\sum_m C_m e^{imGx}.$$

Normal incidence is contained in this expression by setting $k_{x,i}=0$.

<details><summary>Can a tilted plane wave also repeat after one mask period?</summary>

Yes: if $k_{x,i}p=2\pi\ell$ for an integer $\ell$, its complex field repeats after $p$. The unwrapped phase still advances continuously; a wrapped phase plot may show a repeating sawtooth. An arbitrary phase restart at a cell boundary would describe different illumination. For tilted incidence, the phase-removal method keeps order $m$ referenced to the incident wave even in this special commensurate case.

</details>

## 4. Exact analytic check: a smooth mask with exactly three orders

The binary baseline needs infinitely many harmonics. For this final algebra check only, replace it with a different, smooth transmission that has exactly three nonzero coefficients:

$$t(x)=0.6+0.4\cos(Gx)
=0.2e^{-iGx}+0.6+0.2e^{iGx}.$$

With $A_i=1$, $C_{-1}=0.2$, $C_0=0.6$, and $C_{+1}=0.2$. The local product and the order sum are

$$U_{\mathrm{exit}}(x)=[0.6+0.4\cos(Gx)]e^{ik_{x,i}x},$$

$$U_{\mathrm{exit}}(x)
=0.2e^{i(k_{x,i}-G)x}+0.6e^{ik_{x,i}x}+0.2e^{i(k_{x,i}+G)x}.$$

Both calculate the **same complex number at every position**. The code checks that equality, the phase step between cells, and periodicity after removing the incident phase. This example defines the field analytically; no FFT or propagation has been used.

In [ ]:
x_nm = np.linspace(0, 3*p_nm, 1201)
mask = 0.6 + 0.4*np.cos(G*x_nm)
U_i = A_i*np.exp(1j*kx_i*x_nm)
U_exit = mask*U_i

orders = np.array([-1, 0, 1])
t_m = np.array([0.2, 0.6, 0.2])
C_m = A_i*t_m
kx_m = kx_i + orders*G
U_from_orders = np.exp(1j*x_nm[:, None]*kx_m) @ C_m
assert np.allclose(U_exit, U_from_orders, atol=1e-12, rtol=0)

U_next_cell = (0.6+0.4*np.cos(G*(x_nm+p_nm))) * A_i*np.exp(1j*kx_i*(x_nm+p_nm))
assert np.allclose(U_next_cell, np.exp(1j*phase_step)*U_exit, atol=1e-12, rtol=0)
assert np.allclose(U_next_cell*np.exp(-1j*kx_i*(x_nm+p_nm)),
                   U_exit*np.exp(-1j*kx_i*x_nm), atol=1e-12, rtol=0)
print(f'Maximum product/order-sum difference: {np.max(abs(U_exit-U_from_orders)):.2e}')
print('The field changes by the known phase factor across a cell; the periodic factor repeats.')

In [ ]:
display(Image(data=exit_field_reconstruction_png(p_nm, A_i, phase_step)))

The numerical checks compare the prescribed product with its exact finite order sum. The binary mask instead needs infinitely many harmonics; any finite truncation approximates its edges.

[Return to 9.1](../9_1_Defining_the_Complex_Exit_Field.ipynb) | [Sampling in 9.2](../9_2_Sampling_Complex_Fields_and_Their_Spectra.ipynb) | [General Fourier/FFT appendix](../Appendix_4_How_to_Use_FFTs_in_Python.ipynb) | [Unit-cell coefficient calculations](Chapter_9_Unit_Cell_Coefficient_Calculations.ipynb)